In [5]:
from pathlib import Path
import json
import boto3

repo_matches = list(
    Path.home().glob("shared/repos/**/AWS-Lab_SageMaker/devs")
)

if not repo_matches:
    raise FileNotFoundError("Repository AWS-Lab_SageMaker/devs tidak ditemukan")

repo_dir = repo_matches[0]
training_dir = repo_dir / "src" / "training"
train_file = repo_dir / "data" / "customer_churn.example.csv"

assert (training_dir / "train.py").exists()
assert (training_dir / "requirements.txt").exists()
assert train_file.exists()

print("Repository:", repo_dir)
print("Training directory:", training_dir)
print("Dataset:", train_file)

Repository: /home/sagemaker-user/shared/repos/6r04wz32dwpx5u/AWS-Lab_SageMaker/devs
Training directory: /home/sagemaker-user/shared/repos/6r04wz32dwpx5u/AWS-Lab_SageMaker/devs/src/training
Dataset: /home/sagemaker-user/shared/repos/6r04wz32dwpx5u/AWS-Lab_SageMaker/devs/data/customer_churn.example.csv


In [4]:
%cd {repo_dir}
!pytest -q

/mnt/custom-file-systems/s3/shared/repos/6r04wz32dwpx5u/AWS-Lab_SageMaker/devs/src/training
[PosixPath('/mnt/custom-file-systems/s3/shared/repos/6r04wz32dwpx5u/AWS-Lab_SageMaker/devs/src/training/__init__.py'), PosixPath('/mnt/custom-file-systems/s3/shared/repos/6r04wz32dwpx5u/AWS-Lab_SageMaker/devs/src/training/__pycache__'), PosixPath('/mnt/custom-file-systems/s3/shared/repos/6r04wz32dwpx5u/AWS-Lab_SageMaker/devs/src/training/train.py')]


In [ ]:
import sagemaker
from sagemaker.core.helper.session_helper import Session, get_execution_role

aws_session = boto3.Session(region_name="eu-north-1")
region = aws_session.region_name
role = get_execution_role()

account_id = aws_session.client("sts").get_caller_identity()["Account"]
bucket = f"sagemaker-ml-lab-{account_id}-eu-north-1"

print("Region:", region)
print("Account:", account_id)
print("Role:", role)
print("Bucket:", bucket)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Region: ap-southeast-2
Account: 373220261291
Role: arn:aws:iam::373220261291:role/AmazonSageMakerAdminIAMExecutionRole
Bucket: sagemaker-ml-lab-373220261291-ap-southeast-2


In [7]:
s3 = aws_session.client("s3")

s3_key = "datasets/processed/train.csv"
s3.upload_file(
    str(repo_dir / "data" / "customer_churn.example.csv"),
    bucket,
    s3_key,
)

training_data_uri = f"s3://{bucket}/datasets/processed/"
print("Training data:", training_data_uri)

Training data: s3://sagemaker-ml-lab-373220261291-ap-southeast-2/datasets/processed/


In [8]:
response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix="datasets/processed/",
)

for item in response.get("Contents", []):
    print(item["Key"], item["Size"])

datasets/processed/train.csv 290


In [9]:
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import SourceCode, Compute, InputData

trainer = ModelTrainer(
    training_image=training_image,
    role=role,
    source_code=SourceCode(
        source_dir=str(repo_dir),
        entry_script="src/training/train.py",
        requirements="src/training/requirements.txt",
    ),
    compute=Compute(
        instance_type="ml.m5.large",
        instance_count=1,
    ),
)

print("ModelTrainer siap.")

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
ModelTrainer siap.


In [10]:
job = trainer.train(
    input_data_config=[
        InputData(
            channel_name="train",
            data_source=training_data_uri,
        )
    ]
)

print("Training job selesai.")
print(job)

sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱  1 job = trainer.train(                                                                        │
│    2 │   input_data_config=[                                                                     │
│    3 │   │   InputData(                                                                          │
│    4 │   │   │   channel_name="train",                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py:422 in     │
│ wrapper                                                                                          │
│                                                                                                  │
│   419 │   │   │   │   │   caught_ex = e                                                          │
│   420 │   │   │   │   finally:                                                                   │
│   421 │   │   │   │   │   if caught_ex:                                                          │
│ ❱ 422 │   │   │   │   │   │   raise caught_ex                                                    │
│   423 │   │   │   │   │   return response  # pylint: disable=W0150                               │
│   424 │   │   │   else:                                                                          │
│   425 │   │   │   │   logger.debug(                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py:387 in     │
│ wrapper                                                                                          │
│                                                                                                  │
│   384 │   │   │   │   start_timer = perf_counter()                                               │
│   385 │   │   │   │   try:                                                                       │
│   386 │   │   │   │   │   # Call the original function                                           │
│ ❱ 387 │   │   │   │   │   response = func(*args, **kwargs)                                       │
│   388 │   │   │   │   │   stop_timer = perf_counter()                                            │
│   389 │   │   │   │   │   elapsed = stop_timer - start_timer                                     │
│   390 │   │   │   │   │   extra += f"&x-latency={round(elapsed, 2)}"                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:355 in       │
│ wrapper                                                                                          │
│                                                                                                  │
│   352 │   │   │                                                                                  │
│   353 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   354 │   │                                                                                      │
│ ❱ 355 │   │   return run_func(*args, **kwargs)                                                   │
│   356 │                                                                                          │
│   357 │   return wrapper                                                                         │
│   358                                                      

In [ ]:
training_job_name = getattr(job, "job_name", None)

print("Training job:", training_job_name)